# 06 — Linear Forecasting Models (OLS, Ridge, Lasso)

## 1. Introduction and model specification

**Task.** As in Notebook 05: at 12:00 on day $t$, we forecast total electricity consumption in Basel-Stadt on day $d = t + 1$ (MWh/day), using only information available at that time. Following the professor-approved assumption, consumption is treated as known only from day $d-5$ and older (`lag_5` and older).

**Why linear regression?** The lag-7 baseline (Notebook 05) uses a single past value. A linear regression combines several pieces of information available at the cutoff:

- older consumption (`lag_5`, `lag_6`, `lag_7`, `lag_14`, `lag_21`, `lag_28` and the weekly average `lag_mean_5_11`);
- archived weather forecasts for the target day ($W_d$);
- calendar information, including holidays on the target day and on the lagged days ($C_d$).

**Model specification.**

$$
\hat y_d = \beta_0 + \sum_{k \in \{5,6,7,14,21,28\}} \beta_k\, y_{d-k} + \beta_{m}\, \bar y_{d-11:d-5} + \gamma^\top W_d + \delta^\top C_d
$$

Because past values of the target enter as predictors alongside external variables, this is an autoregressive model with exogenous variables (ARX-type). It is not an unrestricted autoregressive model: lags 1–4 are deliberately excluded because they would not be available at the forecasting cutoff.

**Three estimators.**
- **OLS** chooses the coefficients that minimise the sum of squared prediction errors on the training data.
- **Ridge** adds an L2 penalty $\alpha \sum_j \beta_j^2$. It shrinks coefficients towards zero, which stabilises the estimates when predictors are strongly correlated, as our consumption lags are.
- **Lasso** adds an L1 penalty $\alpha \sum_j |\beta_j|$. It can set some coefficients exactly to zero and thereby selects a subset of predictors.

Both penalties depend on the scale of each predictor, so continuous predictors are standardised before fitting. The penalty strength $\alpha$ is chosen by time-ordered cross-validation within each training window (Section 6).

**Evaluation.** All models are evaluated in the 12 monthly validation folds of 2025, on exactly the same 365 target dates as the baseline, using MAE (primary) and RMSE (secondary) in MWh/day. The 2026 holdout is not used in this notebook.

In [1]:
# Setup: imports, project paths and methodology constants
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# A Lasso that has not converged must never go unnoticed: turn the warning into an error
warnings.simplefilter("error", category=ConvergenceWarning)


def find_repo_root(start: Path) -> Path:
    """Walk upwards from `start` until the folder containing data/processed/features.csv is found."""
    for folder in [start, *start.parents]:
        if (folder / "data" / "processed" / "features.csv").exists():
            return folder
    raise FileNotFoundError(
        f"data/processed/features.csv not found above {start}. "
        "Check that you opened the repository, or run code/04_prepare_features.py first."
    )


ROOT = find_repo_root(Path.cwd().resolve())
FEATURES_PATH = ROOT / "data" / "processed" / "features.csv"
FIG_DIR = ROOT / "output" / "figures"
TABLE_DIR = ROOT / "output" / "tables"
BASELINE_PREDICTIONS_PATH = TABLE_DIR / "baseline_validation_predictions.csv"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Methodology constants (identical to Notebook 05 and the approved methodology)
PERIODS = {
    "train": ("2018-01-01", "2024-12-31"),
    "validation": ("2025-01-01", "2025-12-31"),
    "test": ("2026-01-01", "2026-10-31"),
}
MIN_LAG = 5                                   # youngest consumption lag allowed at the cutoff
TRAIN_START = pd.Timestamp(PERIODS["train"][0])
VALIDATION_MONTHS = pd.period_range("2025-01", "2025-12", freq="M")  # the 12 monthly folds

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "| numpy:", np.__version__, "| scikit-learn:", sklearn.__version__)
print("Repository root:", ROOT)

Python: 3.11.9
pandas: 3.1.1 | numpy: 2.4.6 | scikit-learn: 1.9.1
Repository root: /Users/mariegrandfils/Library/CloudStorage/OneDrive-Personal/HSG/Bachelor Economics/DSF/DSF_Group_Project/DSF_forecasting_electricity_consumption_Basel


## 2. Load and check the data

We load `data/processed/features.csv` (prepared by Person 1, not rebuilt here) and repeat the essential structural checks from Notebook 05. We also load the baseline predictions exported by Notebook 05, which are needed for the comparison in Section 9.

In [2]:
# Load the shared feature table (no downloading, no rebuilding) and check its structure
df = pd.read_csv(FEATURES_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)

EXPECTED_COLUMNS = [
    "date", "period", "consumption_mwh", "baseline", "baseline_uses_fallback",
    "weekday", "is_weekend", "month", "is_holiday", "is_christmas_period",
    "lag_7_is_holiday", "lag_14_is_holiday",
    "forecast_temp_mean", "forecast_temp_min", "forecast_temp_max",
    "forecast_precipitation", "heating_degrees", "cooling_degrees",
    "lag_5", "lag_6", "lag_7", "lag_14", "lag_21", "lag_28", "lag_mean_5_11",
]
missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
assert not missing_cols, f"Missing columns: {missing_cols}"

lag_numbers = [int(c[4:]) for c in df.columns if c.startswith("lag_") and c[4:].isdigit()]
assert min(lag_numbers) >= MIN_LAG, f"Lag younger than lag_{MIN_LAG} found"

assert df["date"].is_unique, "Duplicate target dates"
full_calendar = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
assert len(full_calendar.difference(df["date"])) == 0, "Missing calendar days"

for name, (start, end) in PERIODS.items():
    in_window = df["date"].between(pd.Timestamp(start), pd.Timestamp(end))
    assert (df.loc[in_window, "period"] == name).all(), f"Period labels do not match the {name} window"

assert df.isna().sum().sum() == 0, "Missing values found: decide on a treatment before modelling"

print("Shape (rows, columns):", df.shape)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
display(df.groupby("period", sort=False)["date"].agg(first="min", last="max", days="count"))
print("All structural checks passed: columns, lags >= lag_5, unique dates, no gaps, periods, no missing values")

# Baseline predictions from Notebook 05 (needed for the comparison in Section 9)
baseline_pred = pd.read_csv(BASELINE_PREDICTIONS_PATH, parse_dates=["date"])
validation_dates = df.loc[df["period"] == "validation", "date"]
assert len(baseline_pred) == len(validation_dates) == 365, "Baseline file does not have 365 rows"
assert baseline_pred["date"].isin(validation_dates).all(), "Baseline dates differ from the 2025 validation dates"
print(f"Baseline predictions loaded: {len(baseline_pred)} days, "
      f"MAE = {baseline_pred['absolute_error_mwh'].mean():.1f} MWh/day (Notebook 05: 137.3)")

Shape (rows, columns): (3226, 25)
Date range: 2018-01-01 to 2026-10-31


,first,last,days
period,,,
train,2018-01-01,2024-12-31,2557
validation,2025-01-01,2025-12-31,365
test,2026-01-01,2026-10-31,304


All structural checks passed: columns, lags >= lag_5, unique dates, no gaps, periods, no missing values
Baseline predictions loaded: 365 days, MAE = 137.3 MWh/day (Notebook 05: 137.3)


## 3. Choice of explanatory variables

All predictors come from `features.csv` (Person 1); none are created here.

| Group | Columns | Treatment |
|---|---|---|
| Consumption lags | `lag_5`, `lag_6`, `lag_7`, `lag_14`, `lag_21`, `lag_28`, `lag_mean_5_11` | numeric, standardised |
| Weather forecasts | `heating_degrees`, `cooling_degrees`, `forecast_precipitation` | numeric, standardised |
| Calendar (categorical) | `weekday`, `month` | one-hot encoded |
| Calendar (binary) | `is_holiday`, `is_christmas_period`, `lag_7_is_holiday`, `lag_14_is_holiday` | kept as 0/1 |

**Never used as predictors:** `date`, `period`, `consumption_mwh` (the target), and `baseline` / `baseline_uses_fallback`. The baseline is reserved for benchmarking; besides, it equals `lag_7`, which is already included.

**Columns left out to avoid redundant information (verified below):**
- `forecast_temp_mean`: by construction, `heating_degrees − cooling_degrees = 15 − forecast_temp_mean` on every day. With an intercept, OLS cannot estimate separate coefficients for all three (perfect multicollinearity). We keep heating and cooling degrees, because they represent the U-shaped temperature effect found in the exploratory analysis.
- `forecast_temp_min`, `forecast_temp_max`: correlated at about 0.97 with the mean temperature, so they add little information but make individual coefficients unstable.
- `is_weekend`: once `weekday` is one-hot encoded, `is_weekend` equals the sum of the Saturday and Sunday indicators (dummy variable trap).

**Two specifications, fixed in advance.** To see whether the weather forecasts improve predictions, every model is estimated twice:
- `no_weather`: consumption lags + calendar;
- `with_weather`: the same plus the three weather variables.

No further specifications are tried, so model selection stays limited and transparent.

In [3]:
# Feature groups (all prepared by Person 1 in 04_prepare_features.py)
TARGET = "consumption_mwh"
LAG_FEATURES = ["lag_5", "lag_6", "lag_7", "lag_14", "lag_21", "lag_28", "lag_mean_5_11"]
WEATHER_FEATURES = ["heating_degrees", "cooling_degrees", "forecast_precipitation"]
CATEGORICAL_FEATURES = ["weekday", "month"]   # one-hot encoded in Section 4
BINARY_FEATURES = ["is_holiday", "is_christmas_period", "lag_7_is_holiday", "lag_14_is_holiday"]
NEVER_PREDICTORS = ["date", "period", TARGET, "baseline", "baseline_uses_fallback"]

# The two specifications compared on the 2025 validation data (fixed in advance)
SPECS = {
    "no_weather": {"numeric": LAG_FEATURES,
                   "categorical": CATEGORICAL_FEATURES, "binary": BINARY_FEATURES},
    "with_weather": {"numeric": LAG_FEATURES + WEATHER_FEATURES,
                     "categorical": CATEGORICAL_FEATURES, "binary": BINARY_FEATURES},
}


def spec_columns(spec):
    """All input columns of a specification, in a fixed order."""
    return spec["numeric"] + spec["categorical"] + spec["binary"]


# Checks: no forbidden column, no duplicates, no consumption lag younger than lag_5
for name, spec in SPECS.items():
    cols = spec_columns(spec)
    assert not set(cols) & set(NEVER_PREDICTORS), f"'{name}' contains a column that must not be a predictor"
    assert len(cols) == len(set(cols)), f"'{name}' lists a column twice"
    lags = [int(c[4:]) for c in cols if c.startswith("lag_") and c[4:].isdigit()]
    assert min(lags) >= MIN_LAG, f"'{name}' uses a lag younger than lag_{MIN_LAG}"
    print(f"Specification '{name}': {len(cols)} input columns")

# Why some columns are left out: verified on the data
BASE_TEMP = 15.0   # heating/cooling threshold fixed in 04_prepare_features.py
identity_gap = ((df["heating_degrees"] - df["cooling_degrees"]) - (BASE_TEMP - df["forecast_temp_mean"])).abs().max()
print(f"\nheating_degrees − cooling_degrees vs. 15 − forecast_temp_mean: largest difference {identity_gap:.1e} -> exact identity")
weekend_from_weekday = df["weekday"].isin([5, 6]).astype(int)
print("is_weekend equals the Saturday + Sunday dummies on every row:", (weekend_from_weekday == df["is_weekend"]).all())
print("\nCorrelation of the temperature forecasts (training period 2018–2024):")
display(df.loc[df["period"] == "train", ["forecast_temp_mean", "forecast_temp_min", "forecast_temp_max"]].corr().round(2))

Specification 'no_weather': 13 input columns
Specification 'with_weather': 16 input columns

heating_degrees − cooling_degrees vs. 15 − forecast_temp_mean: largest difference 0.0e+00 -> exact identity
is_weekend equals the Saturday + Sunday dummies on every row: True

Correlation of the temperature forecasts (training period 2018–2024):


,forecast_temp_mean,forecast_temp_min,forecast_temp_max
forecast_temp_mean,1.00,0.97,0.97
forecast_temp_min,0.97,1.00,0.89
forecast_temp_max,0.97,0.89,1.00


## 4. Preprocessing

**Two kinds of transformations.**
- *Deterministic feature engineering* (Person 1, `04_prepare_features.py`): lags, holiday flags and heating/cooling degrees follow fixed formulas. Nothing is estimated from the data, so they cannot leak information.
- *Learned preprocessing* (here): imputation medians, standardisation means and standard deviations, and the categories of the one-hot encoder are estimated from data. They must be estimated **only from the training rows of each fold**; otherwise information from the validation period leaks into training.

**Steps, combined in a scikit-learn `Pipeline`:**
- Numeric columns: `SimpleImputer` (median) → `StandardScaler`. There are no missing values in `features.csv` (Section 2); the imputer is a safeguard.
- `weekday` and `month`: `OneHotEncoder(drop="first")`. The first category is dropped to avoid the dummy variable trap with the intercept, so Monday and January are the reference categories. Weekday 0–6 and month 1–12 are categories, not quantities: Sunday (6) is not "six times" Monday.
- Binary flags: kept unchanged as 0/1.

Because the preprocessing and the regression model form one `Pipeline`, calling `fit` on a fold's training rows learns all parameters from those rows only; `predict` then applies them unchanged to the validation month. The same preprocessing is used for OLS, Ridge and Lasso, so the models differ only in their estimator.

In [4]:
def make_preprocessor(spec):
    """Learned preprocessing. Inside a Pipeline it is fitted on the training rows of a fold only."""
    numeric_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),   # safeguard: features.csv has no missing values
        ("scale", StandardScaler()),
    ])
    return ColumnTransformer([
        ("numeric", numeric_steps, spec["numeric"]),
        ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
         spec["categorical"]),
        ("binary", "passthrough", spec["binary"]),
    ])


def make_pipeline(model_name, spec):
    """Preprocessing + regression model in one object: fit() learns everything from the training rows only."""
    models = {
        "ols": LinearRegression(),
        "ridge": Ridge(),
        "lasso": Lasso(max_iter=100_000),
    }
    return Pipeline([("preprocess", make_preprocessor(spec)), ("model", models[model_name])])


# Demonstration on the January 2025 fold: what does the preprocessing learn, and from which rows?
demo_cutoff = VALIDATION_MONTHS[0].start_time - pd.Timedelta(days=MIN_LAG)
demo_train = df[(df["date"] >= TRAIN_START) & (df["date"] <= demo_cutoff)]
demo_spec = SPECS["with_weather"]
demo_pre = make_preprocessor(demo_spec).fit(demo_train[spec_columns(demo_spec)])

feature_names = demo_pre.get_feature_names_out()
print(f"Training rows of the January fold: {len(demo_train)} "
      f"({demo_train['date'].min().date()} to {demo_train['date'].max().date()})")
print("Columns after preprocessing:", len(feature_names))
print(list(feature_names))

scaler = demo_pre.named_transformers_["numeric"].named_steps["scale"]
assert np.isclose(scaler.mean_[0], demo_train["lag_5"].mean())
print(f"\nMean of lag_5 learned by the scaler: {scaler.mean_[0]:.1f} MWh/day "
      f"= mean of lag_5 in the training rows of this fold only")

Training rows of the January fold: 2553 (2018-01-01 to 2024-12-27)
Columns after preprocessing: 31
['numeric__lag_5', 'numeric__lag_6', 'numeric__lag_7', 'numeric__lag_14', 'numeric__lag_21', 'numeric__lag_28', 'numeric__lag_mean_5_11', 'numeric__heating_degrees', 'numeric__cooling_degrees', 'numeric__forecast_precipitation', 'categorical__weekday_1', 'categorical__weekday_2', 'categorical__weekday_3', 'categorical__weekday_4', 'categorical__weekday_5', 'categorical__weekday_6', 'categorical__month_2', 'categorical__month_3', 'categorical__month_4', 'categorical__month_5', 'categorical__month_6', 'categorical__month_7', 'categorical__month_8', 'categorical__month_9', 'categorical__month_10', 'categorical__month_11', 'categorical__month_12', 'binary__is_holiday', 'binary__is_christmas_period', 'binary__lag_7_is_holiday', 'binary__lag_14_is_holiday']

Mean of lag_5 learned by the scaler: 3463.8 MWh/day = mean of lag_5 in the training rows of this fold only
